# LLM Part 1

| § | Slide | What you show |
|---|---|---|
| 0 | — | Setup (run once, before the lecture) |
| 1 | 8  | Generate first text + the three knobs |
| 2 | 37 | Base model vs instruct model |
| 3 | 68, 71 | `input_ids`, then decoding one token at a time |
| 4 | 79 | Tokenizer comparison |
| 5 | 107 | word2vec nearest neighbours + analogy |
| 6 | 113 | Contextual embeddings |
| 7 | 122 | Cosine similarity matrix |

## Setup

In [ ]:
!pip install -q transformers torch sentence-transformers gensim tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 44.1 MB/s eta 0:00:00


In [ ]:
import torch, warnings
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

# Adarsh Dubey --> AdarshDubey [PASCAL CASE]
# Adarsh Dubey --> adarsh_dubey [snake case]

cuda


In [ ]:
# Pre-download everything so nothing stalls mid-lecture.
INSTRUCT = "Qwen/Qwen2.5-0.5B-Instruct"
BASE     = "Qwen/Qwen2.5-0.5B"

_ = AutoTokenizer.from_pretrained(INSTRUCT)
_ = AutoTokenizer.from_pretrained(BASE)
for name in ["bert-base-uncased", "gpt2", "Qwen/Qwen2.5-Coder-0.5B"]:
    _ = AutoTokenizer.from_pretrained(name)
print("tokenizers cached")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

tokenizers cached


## Generate first text  ·  slide 8

Two objects, not one.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT)
model = AutoModelForCausalLM.from_pretrained(INSTRUCT, torch_dtype=torch.float32).to(DEVICE)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

prompt = "what is capital of france ?"

out = generator(prompt, max_new_tokens=120, do_sample=True, return_full_text=True)
print(out[0]["generated_text"])

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


what is capital of france ? 1. The capital of France is Paris.
2. Paris, the second-largest city in France and one of the world's most populous cities, is located on the left bank of the Seine river.
3. It is situated at an elevation of about 350 meters (1,140 feet) above sea level and is surrounded by a green belt that covers over 9 square kilometers (3.5 square miles).
4. The city has a population of approximately 2.1 million people and is home to many famous landmarks such as Notre-Dame Cathedral, the Lou


**Knob 1 — `max_new_tokens`**  ·  slides 10–12

In [ ]:
out = generator(prompt, max_new_tokens=10, do_sample=False, return_full_text=False)
print(repr(out[0]["generated_text"]))

[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


' The capital of France is Paris. It was founded'


**Knob 2 — `do_sample`**  ·  slide 13

In [ ]:
for i in range(3):
    out = generator(prompt, max_new_tokens=15, do_sample=False, return_full_text=False)
    print(i, out[0]["generated_text"].strip()[:60])

[transformers] Both `max_new_tokens` (=15) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=15) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0 The capital of France is Paris. It was founded in 987


[transformers] Both `max_new_tokens` (=15) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1 The capital of France is Paris. It was founded in 987
2 The capital of France is Paris. It was founded in 987


In [ ]:
for i in range(3):
    out = generator(prompt, max_new_tokens=20, do_sample=True, temperature=1.0, return_full_text=False)
    print(i, out[0]["generated_text"].strip()[:60])

[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0 The capital of France is Paris. It was founded in 1250 as a 


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1 The capital of France is Paris. 

Does it follow that "Paris
2 The capital of France is Paris.

is it right to say that you


**Knob 3 — `return_full_text`**  ·  slide 14

In [ ]:
out = generator(prompt, max_new_tokens=15, do_sample=False, return_full_text=True)
print(out[0]["generated_text"])

[transformers] Both `max_new_tokens` (=15) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


what is capital of france ? The capital of France is Paris. It was founded in 987


> ⏎ Back to slides — **slide 15**

---
## § 2 — Base vs instruct  ·  slide 37

Same family, same size. Only difference is phase 2.

In [ ]:
base_tok = AutoTokenizer.from_pretrained(BASE)
base_model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to(DEVICE)
base_gen = pipeline("text-generation", model=base_model, tokenizer=base_tok)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [ ]:
question = "What is the capital of France         ?"

print("--- BASE ---")
print(base_gen(question, max_new_tokens=60, do_sample=False, return_full_text=False)[0]["generated_text"])

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- BASE ---
 Paris


In [ ]:
chat = tokenizer.apply_chat_template(
    [{"role": "user", "content": question}],
    tokenize=False, add_generation_prompt=True
)

print("--- INSTRUCT ---")
print(generator(chat, max_new_tokens=60, do_sample=False, return_full_text=False)[0]["generated_text"])

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- INSTRUCT ---
The capital of France is Paris.


Optional — show what the chat template actually wraps around the question:

In [ ]:
print(chat)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France         ?<|im_end|>
<|im_start|>assistant



> ⏎ Back to slides — **slide 38**

---
## § 3 — Inside the tokenizer  ·  slides 68 and 71

### 3a — `input_ids`  ·  slide 68

In [ ]:
prompt = "    hey!@1`"


ids = tokenizer(prompt).input_ids
print(len(ids))
print(ids)

# robot + icss
# robotics + ss
# [220, 220]

6
[262, 34209, 0, 31, 16, 63]


> ⏎ Back to slides — **slides 69, 70**

### 3b — decode one at a time  ·  slide 71

In [ ]:
for i in ids:
    print(f"{i:>7}  |{tokenizer.decode([i])}|")

    262  |   |
  34209  | hey|
      0  |!|
     31  |@|
     16  |1|
     63  |`|


Words arrive in pieces — slides 73, 74:

In [ ]:
for word in ["Subject", "unbelievable", "tokenization", "Priya", "Subjective"]:
    pieces = [tokenizer.decode([i]) for i in tokenizer(word).input_ids]
    print(f"{word:>15}  ->  {pieces}")

        Subject  ->  ['Subject']
   unbelievable  ->  ['un', 'belie', 'vable']
   tokenization  ->  ['token', 'ization']
          Priya  ->  ['Pri', 'ya']
     Subjective  ->  ['Subject', 'ive']


> ⏎ Back to slides — **slide 72**

---
## § 4 — Tokenizer comparison  ·  slide 79

Coloured blocks = token boundaries.

In [ ]:
import tiktoken

BG = ["\033[48;5;153m", "\033[48;5;223m", "\033[48;5;157m", "\033[48;5;218m"]
RESET = "\033[0m"

_hf_cache = {}

def pieces(name, text):
    if name == "gpt-4":
        enc = tiktoken.get_encoding("cl100k_base")
        return [enc.decode([i]) for i in enc.encode(text)]
    if name not in _hf_cache:
        _hf_cache[name] = AutoTokenizer.from_pretrained(name)
    tok = _hf_cache[name]
    return [tok.decode([i]) for i in tok(text, add_special_tokens=False).input_ids]

def show(name, text):
    p = pieces(name, text)
    blocks = "".join(BG[i % len(BG)] + s.replace("\n", "\\n") + RESET for i, s in enumerate(p))
    print(f"{name:<28} {len(p):>3} tokens   {blocks}")

TOKENIZERS = ["bert-base-uncased", "gpt2", "gpt-4",
              "Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-Coder-0.5B"]

**The test string**  ·  slides 77, 80–84

In [ ]:
TEST = "English and CAPITALIZATION 🎵 show_tokens False None elif == >= 12.0*50=600"

for name in TOKENIZERS:
    show(name, TEST)

bert-base-uncased             24 tokens   englishandcapital##ization[UNK]show_token##sfalsenoneeli##f==>=12.0*50=600
gpt2                          27 tokens   English and CAPITALIZATION ��� show_tokens False None elif == >= 12.0*50=600
gpt-4                         22 tokens   English and CAPITALIZATION ��� show_tokens False None elif == >= 12.0*50=600
Qwen/Qwen2.5-0.5B             26 tokens   English and CAPITALIZATION ��� show_tokens False None elif == >= 12.0*50=600
Qwen/Qwen2.5-Coder-0.5B       26 tokens   English and CAPITALIZATION ��� show_tokens False None elif == >= 12.0*50=600


**Indentation**  ·  slide 85

In [ ]:
CODE = 'def add(a, b):\n    """Add two numbers."""\n    return a + b'

for name in TOKENIZERS:
    show(name, CODE)

bert-base-uncased             22 tokens   defadd(a,b):"""addtwonumbers."""returna+b
gpt2                          25 tokens   def add(a, b):\n    """Add two numbers."""\n    return a + b
gpt-4                         17 tokens   def add(a, b):\n    """Add two numbers."""\n    return a + b
Qwen/Qwen2.5-0.5B             17 tokens   def add(a, b):\n    """Add two numbers."""\n    return a + b
Qwen/Qwen2.5-Coder-0.5B       17 tokens   def add(a, b):\n    """Add two numbers."""\n    return a + b


**Non-English**  ·  slide 86

In [ ]:
EN = "The train to the city leaves early in the morning."
HI = "शहर जाने वाली ट्रेन सुबह जल्दी निकलती है।"

for name in ["gpt2", "gpt-4", "Qwen/Qwen2.5-0.5B"]:
    a, b = len(pieces(name, EN)), len(pieces(name, HI))
    print(f"{name:<24}  en {a:>3}   hi {b:>3}   x{b/a:.1f}")

gpt2                      en  11   hi  66   x6.0
gpt-4                     en  11   hi  42   x3.8
Qwen/Qwen2.5-0.5B         en  11   hi  39   x3.5


**The emoji**  ·  slide 87

In [ ]:
for name in ["gpt2", "gpt-4"]:
    print(f"{name:<10}", pieces(name, "🎵 hey"))

gpt2       ['�', '�', '�', ' hey']
gpt-4      ['�', '�', '�', ' hey']


**Whitespace changes the IDs**  ·  slide 95

In [ ]:
for text in ["The answer is  9", "The answer is "]:
    print(repr(text), "->", tokenizer(text, add_special_tokens=False).input_ids)

'The answer is  9' -> [785, 4226, 374, 220, 220, 24]
'The answer is ' -> [785, 4226, 374, 220]


**Letters are invisible inside a token**  ·  slide 93

In [ ]:
print(pieces("gpt-4", "strawberry 9"))
print(pieces("gpt-4", "9.11 vs 9.9"))

['str', 'aw', 'berry', ' ', '9']
['9', '.', '11', ' vs', ' ', '9', '.', '9']


> ⏎ Back to slides — **slide 80**

---
## § 5 — word2vec nearest neighbours  ·  slide 107

In [ ]:
import gensim.downloader as api

wv = api.load("glove-wiki-gigaword-100")   # ~130 MB

[==================================================] 100.0% 128.1/128.1MB downloaded


KeyboardInterrupt: 

In [ ]:
for word in ["king", "python", "mumbai"]:
    print(word)
    for w, s in wv.most_similar(word, topn=5):
        print(f"    {s:.3f}  {w}")

**The analogy**  ·  slide 109

In [ ]:
print(wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3))
print(wv.most_similar(positive=["paris", "italy"], negative=["france"], topn=3))

**One vector, forever**  ·  slides 110, 111

In [ ]:
print(wv["bank"][:8])
print("river/bank ", wv.similarity("bank", "river"))
print("money/bank ", wv.similarity("bank", "money"))

> ⏎ Back to slides — **slide 108**

---
## § 6 — Contextual embeddings  ·  slide 113

In [ ]:
ctx_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
ctx_model = AutoModel.from_pretrained("bert-base-uncased").to(DEVICE).eval()

def token_vectors(text):
    enc = ctx_tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        h = ctx_model(**enc).last_hidden_state[0]
    toks = ctx_tok.convert_ids_to_tokens(enc.input_ids[0])
    return toks, h

In [ ]:
toks, h = token_vectors("The vet examined the dog.")
print(h.shape)
for t, v in zip(toks, h):
    print(f"{t:<12} {[round(x, 2) for x in v[:6].tolist()]}")

**Same word, different vectors**  ·  slide 115

In [ ]:
import torch.nn.functional as F

S1 = "He sat on the river bank and watched the water."
S2 = "She walked to the bank to deposit a cheque."
S3 = "We fished from the muddy bank all afternoon."

def vector_for(word, sentence):
    toks, h = token_vectors(sentence)
    return h[toks.index(word)]

b1, b2, b3 = vector_for("bank", S1), vector_for("bank", S2), vector_for("bank", S3)

def cos(a, b):
    return round(F.cosine_similarity(a, b, dim=0).item(), 3)

print("river-bank  vs  money-bank :", cos(b1, b2))
print("river-bank  vs  river-bank :", cos(b1, b3))
print()
print("context separated them:", cos(b1, b3) > cos(b1, b2))

> ⏎ Back to slides — **slide 114**

---
## § 7 — Cosine similarity matrix  ·  slide 122

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
SENTENCES = [
    "She took her puppy to see a vet.",                # A
    "The dog was examined by an animal doctor.",       # B  same meaning as A, ZERO shared words
    "The bank on the corner raised interest rates.",   # C
    "The bank on the river flooded after heavy rain.", # D  many shared words with C, different meaning
]
LABELS = ["A", "B", "C", "D"]

for l, s in zip(LABELS, SENTENCES):
    print(l, s)

**What word-matching says**  ·  slide 121

In [ ]:
import numpy as np, re

def bag(s):
    return set(re.findall(r"[a-z]+", s.lower()))

def jaccard(a, b):
    A, B = bag(a), bag(b)
    return len(A & B) / len(A | B)

overlap = np.array([[jaccard(a, b) for b in SENTENCES] for a in SENTENCES])

def grid(m, title):
    print(title)
    print("      " + "".join(f"{l:>7}" for l in LABELS))
    for l, row in zip(LABELS, m):
        print(f"{l:>4}  " + "".join(f"{v:>7.2f}" for v in row))

grid(overlap, "word overlap")

**What embeddings say**  ·  slides 123, 124

In [ ]:
E = embedder.encode(SENTENCES, normalize_embeddings=True)
cos = E @ E.T

grid(cos, "cosine similarity")

In [ ]:
print(f"A–B   words {overlap[0,1]:.2f}   meaning {cos[0,1]:.2f}   (same thing, no shared words)")
print(f"C–D   words {overlap[2,3]:.2f}   meaning {cos[2,3]:.2f}   (shared words, different thing)")
print()
print("word matching ranks C–D higher:", overlap[2,3] > overlap[0,1])
print("embeddings rank A–B higher:    ", cos[0,1] > cos[2,3])

**The RAG loop in four lines**  ·  slides 125–127

In [ ]:
DOCS = [
    "Refunds are processed within 5 business days.",
    "Our office is open from 9am to 6pm on weekdays.",
    "You can reset your password from the account settings page.",
    "Shipping is free on orders above 2000 rupees.",
]

D = embedder.encode(DOCS, normalize_embeddings=True)
q = embedder.encode("I forgot my login details", normalize_embeddings=True)

scores = D @ q
for i in np.argsort(-scores):
    print(f"{scores[i]:.3f}  {DOCS[i]}")

**Any sequence works**  ·  slides 129, 130

In [ ]:
from gensim.models import Word2Vec

playlists = [
    ["Bohemian Rhapsody", "Stairway to Heaven", "Hotel California", "Sweet Child O Mine"],
    ["Stairway to Heaven", "Hotel California", "Comfortably Numb", "Bohemian Rhapsody"],
    ["Blinding Lights", "Levitating", "Save Your Tears", "Don't Start Now"],
    ["Levitating", "Don't Start Now", "Blinding Lights", "As It Was"],
    ["Hotel California", "Comfortably Numb", "Sweet Child O Mine", "Stairway to Heaven"],
    ["Save Your Tears", "As It Was", "Blinding Lights", "Levitating"],
] * 40

song2vec = Word2Vec(playlists, vector_size=32, window=3, min_count=1,
                    epochs=60, sg=1, seed=0, workers=1)

for s, sc in song2vec.wv.most_similar("Hotel California", topn=3):
    print(f"{sc:.3f}  {s}")

> ⏎ Back to slides — **slide 123**. Done.